In [ ]:
# lightning_mobilenet_cifar_mlflow.py
import argparse, os, torch
import lightning as L
from torch import nn
from torch.utils.data import random_split, DataLoader
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torchvision.models import mobilenet_v3_small
import mlflow.pytorch
from mlflow import MlflowClient
import torchmetrics
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, StochasticWeightAveraging
# from pytorch_lightning.callbacks.progress import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger, WandbLogger, MLFlowLogger
from lightning.pytorch.tuner import Tuner
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

In [ ]:
# ------------- Data ----------------
class CIFAR10DataModule(L.LightningDataModule):
    def __init__(self, data_dir="../data", batch_size=128, num_workers=4):
        super().__init__()
        self.save_hyperparameters()
        self.train_t = T.Compose([
            T.Resize(224), T.RandomHorizontalFlip(),
            T.RandAugment(num_ops = 5),
            T.ToTensor(),
            T.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)),
            T.RandomErasing(),
        ])
        self.test_t = T.Compose([
            T.Resize(224), T.ToTensor(),
            T.Normalize((0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)),
        ])

    def prepare_data(self):
        CIFAR10(self.hparams.data_dir, train=True, download=True)
        CIFAR10(self.hparams.data_dir, train=False, download=True)

    def setup(self, stage=None):
        full = CIFAR10(self.hparams.data_dir, train=True, transform=self.train_t)
        self.train_set, self.val_set = random_split(full, [45_000, 5_000])
        self.test_set = CIFAR10(self.hparams.data_dir, train=False, transform=self.test_t)

    def _loader(self, ds, shuffle):
        return DataLoader(ds, batch_size=self.hparams.batch_size,
                          shuffle=shuffle, num_workers=self.hparams.num_workers,
                          pin_memory=True)

    def train_dataloader(self): return self._loader(self.train_set, True)
    def val_dataloader(self):   return self._loader(self.val_set, False)
    def test_dataloader(self):  return self._loader(self.test_set, False)


In [ ]:
# ------------- Model ---------------
class MobileNetCIFAR(L.LightningModule):
    def __init__(self, num_classes=10, lr=3e-4, freeze_backbone=True):
        super().__init__()
        self.save_hyperparameters()

        backbone = mobilenet_v3_small(weights="DEFAULT")
        if freeze_backbone:
            for p in backbone.parameters():
                p.requires_grad = False

        in_features = backbone.classifier[3].in_features
        backbone.classifier[3] = nn.Linear(in_features, num_classes)
        self.model = backbone

        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc  = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x): return self.model(x)

    def _shared_step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss   = self.criterion(logits, y)
        preds  = logits.argmax(1)
        getattr(self, f"{stage}_acc")(preds, y)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_acc", getattr(self, f"{stage}_acc"), prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, b, i):  return self._shared_step(b, "train")
    def validation_step(self, b, i): self._shared_step(b, "val")
    def test_step(self, b, i):       self._shared_step(b, "test")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
        return [opt], [sch]

In [ ]:
def print_auto_logged_info(r):
    tags = {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.")}
    artifacts = [f.path for f in MlflowClient().list_artifacts(r.info.run_id, "model")]
    print(f"run_id: {r.info.run_id}")
    print(f"artifacts: {artifacts}")
    print(f"params: {r.data.params}")
    print(f"metrics: {r.data.metrics}")
    print(f"tags: {tags}")

In [ ]:
batch_size=128
lr=3e-4
max_epochs=20
freeze_backbone=False
tracking_uri="mlruns/base_model_mobilenet"

In [ ]:
# import tempfile, os
# mlflow.set_tracking_uri(f"file:{os.getcwd()}/clean_mlruns")   # brand-new path

# mlf_logger = MLFlowLogger(
#     experiment_name="cifar10_mobilenet_v3",
#     tracking_uri=mlflow.get_tracking_uri(),
#     log_model=True,
# )

In [ ]:
L.seed_everything(42, workers=True)

# Create the experiment
# experiment_id = mlflow.create_experiment(name="MobileNet-Exp", artifact_location=tracking_uri)

mlflow.set_tracking_uri(tracking_uri)

# experiment = mlflow.entities.Experiment(artifact_location='mlruns/737497459240027344',
# creation_time= 1747159570234,
# experiment_id= '737497459240027344',
# last_update_time = 1747159570234,
# lifecycle_stage= 'active',
# name="cifar10_mobilenet_v3")
# mlflow.set_experiment(experiment_name='test')

In [ ]:
mlf_logger = MLFlowLogger(
        experiment_name="cifar10_mobilenet_v3",
        tracking_uri=mlflow.get_tracking_uri(),
        log_model=True,                           # auto-logs best checkpoint
        tags={"model": "mobilenet_v3_small"}
    )

In [ ]:
dm = CIFAR10DataModule(batch_size=batch_size)
model = MobileNetCIFAR(lr=lr, freeze_backbone=freeze_backbone)
ckpt_cb = ModelCheckpoint(monitor="val_acc",  dirpath=tracking_uri, filename="{epoch}-{val_loss:.2f}", mode="max", save_top_k=1)

In [ ]:
trainer = L.Trainer(
        enable_progress_bar=True,
        max_epochs=max_epochs,
        accelerator="gpu",
        devices=4,
        precision="bf16-mixed" if torch.cuda.is_available() else "32-true",
        logger=mlf_logger,
        callbacks=[ckpt_cb, 
                   StochasticWeightAveraging(swa_epoch_start=0.8, swa_lrs=0.1, 
                                             annealing_epochs=10, annealing_strategy='cos', 
                                             avg_fn=None, device='cuda')],
        log_every_n_steps=25,
    )

In [ ]:
mlflow.pytorch.autolog()
with mlflow.start_run() as run:
    trainer.fit(model, dm)

print_auto_logged_info(mlflow.get_run(run_id=run.info.run_id))

# trainer.test(model, datamodule=dm)

In [ ]:
import os

import lightning as L
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchmetrics import Accuracy
from torchvision import transforms
from torchvision.datasets import MNIST

import mlflow.pytorch
from mlflow import MlflowClient


class MNISTModel(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(28 * 28, 10)
        self.accuracy = Accuracy("multiclass", num_classes=10)

    def forward(self, x):
        return torch.relu(self.l1(x.view(x.size(0), -1)))

    def training_step(self, batch, batch_nb):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        pred = logits.argmax(dim=1)
        acc = self.accuracy(pred, y)

        # PyTorch `self.log` will be automatically captured by MLflow.
        self.log("train_loss", loss, on_epoch=True)
        self.log("acc", acc, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.02)


def print_auto_logged_info(r):
    tags = {k: v for k, v in r.data.tags.items() if not k.startswith("mlflow.")}
    artifacts = [f.path for f in MlflowClient().list_artifacts(r.info.run_id, "model")]
    print(f"run_id: {r.info.run_id}")
    print(f"artifacts: {artifacts}")
    print(f"params: {r.data.params}")
    print(f"metrics: {r.data.metrics}")
    print(f"tags: {tags}")


# Initialize our model.
mnist_model = MNISTModel()

# Load MNIST dataset.
train_ds = MNIST(
    os.getcwd(), train=True, download=True, transform=transforms.ToTensor()
)
# Only take a subset of the data for faster training.
indices = torch.arange(32)
train_ds = Subset(train_ds, indices)
train_loader = DataLoader(train_ds, batch_size=8)

# Initialize a trainer.
trainer = L.Trainer(max_epochs=3)

# Auto log all MLflow entities
mlflow.pytorch.autolog()

# Train the model.
with mlflow.start_run() as run:
    trainer.fit(mnist_model, train_loader)

# Fetch the auto logged parameters and metrics.
print_auto_logged_info(mlflow.get_run(run_id=run.info.run_id))
